# `sovereign_spread(country=None, on=None, revision=None)`

The extra yield each government's debt pays over a riskless benchmark for its default risk, from `macro.damodaran_country_risk`. Two measures per row: `default_spread` from the country's Moody's rating (or PRS score), never NULL, and `cds_spread_net`, the 10-year sovereign credit default swap spread net of the edition's benchmark country, floored at 0 and NULL where no CDS trades.

Every cell runs against prod, read-only, as `readonly_user`. Change the arguments and re-run any cell.

`country` is an ISO alpha-3 code (`AE-AZ`, `AE-RK`, `AE-SH` for the three emirates) or a name from that edition, in any case and spacing; `None` gives every country. `on` picks the edition in force on that date (default: today in US/Eastern), and `revision` defaults to its newest. Values are Damodaran's, as published and never recomputed, in fractions: 0.0442 is 4.42%.

In [1]:
from datetime import date

import polars as pl
from IPython.display import Markdown

from arora_data.sovereign_rating import SovereignRating
from arora_data.sovereign_spread import SovereignSpread
from arora_data.sync import damodaran_editions, sovereign_rating, sovereign_spread
from arora_data.dates import today_et
from arora_data.frames import to_df


def show(rows):
    """One result or a list of results as a Markdown table."""
    rows = [r.model_dump() for r in (rows if isinstance(rows, list) else [rows])]
    if not rows:
        return Markdown("0 rows")
    head = list(rows[0])
    lines = ["|".join(head), "|".join("---" for _ in head)]
    lines += ["|".join(str(r[c]).replace("|", "/") for c in head) for r in rows]
    return Markdown(f"{len(rows)} rows\n\n" + "\n".join(lines))

## One country, by code or name

In [2]:
assert sovereign_spread("BRA") == sovereign_spread("brazil")
show(sovereign_spread("BRA"))

1 rows

edition_date|revision|entity_code|published_name|entity_type|region|default_spread|cds_spread_net
---|---|---|---|---|---|---|---
2026-07-01|1|BRA|Brazil|country|Central and South America|0.0199318022|0.0208000000

## His spread for each rating
Every country with the same rating gets the same default spread, from his table of spreads by rating.

In [3]:
spread = to_df(sovereign_spread(), SovereignSpread)
rating = to_df(sovereign_rating(), SovereignRating)
both = spread.join(rating.select("entity_code", "basis", "moodys_rating"), on="entity_code")
(both.filter(pl.col("basis") == "rating").group_by("moodys_rating")
 .agg(pl.col("default_spread").min().alias("spread"),
      pl.col("default_spread").n_unique().alias("distinct spreads"), pl.len().alias("countries"))
 .sort("spread"))

moodys_rating,spread,distinct spreads,countries
str,"decimal[38,10]",u32,u32
"""Aaa""",0.0000000000,1,12
"""Aa1""",0.0021863537,1,3
"""Aa2""",0.0039302145,1,4
"""Aa3""",0.0047724033,1,11
"""A1""",0.0056145922,1,8
…,…,…,…
"""Caa1""",0.0597018300,1,14
"""Caa2""",0.0716796266,1,7
"""Caa3""",0.0796336322,1,4


## The CDS-based measure
Only countries with a traded CDS have one. It is net of the benchmark: the US's through the 2025-01-01 edition, Switzerland's after. Below the benchmark it is 0.

In [4]:
print("with a CDS:", spread["cds_spread_net"].is_not_null().sum(), "of", spread.height)
print({e.edition_date.year: e.cds_benchmark for e in damodaran_editions() if e.latest})
spread.drop_nulls("cds_spread_net").sort("cds_spread_net").head(8).select("entity_code", "published_name", "default_spread", "cds_spread_net")

with a CDS: 78 of 178


{2015: 'US', 2016: 'US', 2017: 'US', 2018: 'US', 2019: 'US', 2020: 'US', 2021: 'US', 2022: 'US', 2023: 'US', 2024: 'US', 2025: 'Switzerland', 2026: 'Switzerland'}


entity_code,published_name,default_spread,cds_spread_net
str,str,"decimal[38,10]","decimal[38,10]"
"""CHE""","""Switzerland""",0.0000000000,0.0000000000
"""NOR""","""Norway""",0.0000000000,0.0002000000
"""SWE""","""Sweden""",0.0000000000,0.0005000000
"""DEU""","""Germany""",0.0000000000,0.0006000000
"""DNK""","""Denmark""",0.0000000000,0.0006000000
"""AUS""","""Australia""",0.0000000000,0.0007000000
"""NLD""","""Netherlands""",0.0000000000,0.0007000000
"""FIN""","""Finland""",0.0021863537,0.0011000000


## Known source issue
In the 2026-07-01 edition Barbados's Moody's rating is typed as C in his file, so its default spread is C's, 17.50%, the same as Belarus.

In [5]:
show(sovereign_spread("BRB", on=date(2026, 4, 1)) + sovereign_spread("BRB", on=date(2026, 7, 1)))

2 rows

edition_date|revision|entity_code|published_name|entity_type|region|default_spread|cds_spread_net
---|---|---|---|---|---|---|---
2026-04-01|1|BRB|Barbados|country|Caribbean|0.0523432098|None
2026-07-01|1|BRB|Barbados|country|Caribbean|0.1750000000|None

## Bad input

In [6]:
for kwargs in [dict(country="South Korea"), dict(country="US"), dict(on=date(2014, 12, 31)),
               dict(on=date(2026, 3, 31), revision=7)]:
    try:
        sovereign_spread(**kwargs)
    except ValueError as e:
        print(e)

'South Korea' is not a stored country in Damodaran's 2026-07-01 edition: use an ISO alpha-3 code or a name from that edition, such as USA or 'United States'


'US' is not a stored country in Damodaran's 2026-07-01 edition: use an ISO alpha-3 code or a name from that edition, such as USA or 'United States'


no Damodaran edition on or before 2014-12-31: the first is 2015-01-01


Damodaran's 2026-01-01 edition has no revision 7: it has 1, 2, 3, 4
